# Run All Implemented Phases

This notebook is the main entry point for the implemented pipeline. It validates existing Phase 1, Phase 2, and Phase 3 artifacts, and only reruns expensive steps when their outputs are missing or when the force flags are enabled.

In [ ]:
# Cell 1 - Set up imports and make the project package importable from notebooks.
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


In [ ]:
# Cell 2 - Load the shared YAML config and resolve the main artifact paths.
from src.data_loading import load_config, resolve_project_path

config = load_config(CONFIG_PATH)
processed_dir = resolve_project_path(config["paths"]["processed_dir"], PROJECT_ROOT)
models_dir = resolve_project_path(config["paths"]["models_dir"], PROJECT_ROOT)
metrics_dir = resolve_project_path(config["paths"]["metrics_dir"], PROJECT_ROOT)
figures_dir = resolve_project_path(config["paths"]["figures_dir"], PROJECT_ROOT)

print(f"Expected feature count: {config['data']['expected_feature_count']}")
print(f"Expected classes: {config['data']['expected_num_classes']}")


In [ ]:
# Cell 3 - Run Phase 1 inspection only when canonical metric outputs are missing.
from src.data_loading import inspect_dataset

phase1_outputs = [
    metrics_dir / "class_distribution.csv",
    metrics_dir / "feature_summary.csv",
    metrics_dir / "dropped_features.csv",
    metrics_dir / "quasi_constant_review.csv",
    metrics_dir / "data_inspection_report.txt",
]

if all(path.exists() for path in phase1_outputs):
    class_distribution = pd.read_csv(metrics_dir / "class_distribution.csv")
    print("Phase 1 outputs already exist; loaded class distribution from disk.")
    print(f"Rows in distribution table: {len(class_distribution)}")
else:
    phase1_report = inspect_dataset(CONFIG_PATH)
    print("Phase 1 inspection completed.")
    print(phase1_report)


In [ ]:
# Cell 4 - Run Phase 2 preprocessing only when processed arrays or the report are missing.
from src.preprocessing import preprocess_dataset

phase2_outputs = [
    processed_dir / "X_train.npy",
    processed_dir / "X_test.npy",
    processed_dir / "y_train.npy",
    processed_dir / "y_test.npy",
    processed_dir / "minmax_scaler.joblib",
    processed_dir / "label_mapping.json",
    metrics_dir / "preprocessing_report.json",
    metrics_dir / "split_class_distribution.csv",
]

FORCE_REBUILD_PHASE2 = False
if FORCE_REBUILD_PHASE2 or not all(path.exists() for path in phase2_outputs):
    phase2_report = preprocess_dataset(CONFIG_PATH)
else:
    with open(metrics_dir / "preprocessing_report.json", "r", encoding="utf-8") as file:
        phase2_report = json.load(file)

print("Phase 2 report:")
print(json.dumps(phase2_report, indent=2))


In [ ]:
# Cell 5 - Run Phase 3 Autoencoder training only when model artifacts are missing.
from src.autoencoder import Autoencoder, train_autoencoder
import torch

phase3_outputs = [
    models_dir / "autoencoder.pt",
    models_dir / "encoder.pt",
    metrics_dir / "autoencoder_history.csv",
    metrics_dir / "autoencoder_reconstruction_error.json",
    figures_dir / "ae_loss_curve.png",
]

FORCE_RETRAIN_PHASE3 = False
if FORCE_RETRAIN_PHASE3 or not all(path.exists() for path in phase3_outputs):
    phase3_report = train_autoencoder(CONFIG_PATH, device_name="auto")
else:
    with open(metrics_dir / "autoencoder_reconstruction_error.json", "r", encoding="utf-8") as file:
        phase3_report = json.load(file)

print("Phase 3 reconstruction report:")
print(json.dumps(phase3_report, indent=2))

# Load the checkpoint and run a tiny encoder smoke test.
def load_torch_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

checkpoint = load_torch_checkpoint(models_dir / "autoencoder.pt")
metadata = checkpoint["metadata"]
model = Autoencoder(input_dim=metadata["input_dim"], latent_dim=metadata["latent_dim"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

x_test = np.load(processed_dir / "X_test.npy", mmap_mode="r")
with torch.no_grad():
    sample = torch.as_tensor(np.array(x_test[:5], dtype=np.float32, copy=True))
    latent = model.encoder(sample)
    reconstruction = model(sample)

print(f"Sample input shape: {tuple(sample.shape)}")
print(f"Latent shape: {tuple(latent.shape)}")
print(f"Reconstruction shape: {tuple(reconstruction.shape)}")


In [ ]:
# Cell 6 - Summarize the implemented pipeline artifacts in one table.
summary_rows = [
    {"phase": "Phase 1", "artifact": "class_distribution.csv", "exists": (metrics_dir / "class_distribution.csv").exists()},
    {"phase": "Phase 1", "artifact": "feature_summary.csv", "exists": (metrics_dir / "feature_summary.csv").exists()},
    {"phase": "Phase 2", "artifact": "X_train.npy", "exists": (processed_dir / "X_train.npy").exists()},
    {"phase": "Phase 2", "artifact": "X_test.npy", "exists": (processed_dir / "X_test.npy").exists()},
    {"phase": "Phase 3", "artifact": "autoencoder.pt", "exists": (models_dir / "autoencoder.pt").exists()},
    {"phase": "Phase 3", "artifact": "encoder.pt", "exists": (models_dir / "encoder.pt").exists()},
    {"phase": "Phase 3", "artifact": "ae_loss_curve.png", "exists": (figures_dir / "ae_loss_curve.png").exists()},
]
pd.DataFrame(summary_rows)
